# Fine-Tuning Qwen2.5-1.5B-Instruct untuk Insight Agent Domain Energi (QLoRA + Unsloth)

Proyek: `local-agentic-analytics` — Tugas Akhir Yoga Firman Syahputra

**Tujuan:** Melatih narrator Qwen dengan resep IDENTIK dengan `finetune-gemma-energy.ipynb`, untuk komparasi apple-to-apple kualitas narasi report LaTeX/PDF (gemma2-energy-insight vs qwen25-energy-insight), yang nanti dinilai dengan checker deterministik + LLM-as-a-judge (lihat `docs/llm_judge_report_eval_plan.md`).

**Kenapa Qwen2.5-1.5B (bukan 3B):** pemenang benchmark internal (akurasi vs baseline p<0,001, VRAM 1126 MB, latency terendah) dan berlisensi **Apache-2.0** (3B memakai Qwen Research License non-komersial). Lihat `docs/model_selection.md`.

**Aturan apple-to-apple (dari `docs/finetune_recipe.md`) — JANGAN diubah:**
- Dataset identik dengan run gemma: `train.jsonl` (194) + `val.jsonl` (49), hash dicatat di Bagian 2.
- LoRA identik: r=16, alpha=16, dropout=0, target modules sama, seed 42.
- Training identik: 3 epoch, batch efektif 8 (2×4), LR 2e-4, max_seq 1024.
- Export identik: merge → GGUF → Q4_K_M; system prompt Modelfile identik dengan deploy gemma.

**Alur:** 1) Setup GPU → 2) Dataset + hash → 3) Load Qwen 4-bit (dengan penanganan Log/Warning) → 4) Chat template (ChatML) → 5) Training QLoRA → 6) Sanity check anti-halusinasi → 7) Simpan adapter + GGUF → 8) Deploy Ollama → 9) Evaluasi.

> Catatan tesis: training di Kaggle/Colab tidak melanggar premis *local inference*; yang lokal adalah deployment GGUF hasil training pada GTX 1650.

## 1. Setup & Cek GPU

Verifikasi T4 aktif sebelum instalasi.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 2. Dataset (IDENTIK dengan run gemma) + Catat Hash

Gunakan `train.jsonl` dan `val.jsonl` yang SAMA PERSIS dengan yang dipakai fine-tune gemma. Hash SHA256 di bawah harus sama dengan yang tercatat di `docs/finetune_recipe.md` — kalau beda, perbandingan antar-kandidat tidak valid.

In [6]:
import os
import json
import hashlib

# Sesuaikan dengan direktori dataset Kaggle-mu (sama seperti run gemma).
dataset_dir = "/kaggle/input/datasets/yogafsyahputra/finetune"

train_path = os.path.join(dataset_dir, "train.jsonl")
val_path = os.path.join(dataset_dir, "val.jsonl")

assert os.path.exists(train_path), f"File tidak ditemukan di: {train_path}"
assert os.path.exists(val_path), f"File tidak ditemukan di: {val_path}"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

print("SHA256 train.jsonl:", sha256_file(train_path))
print("SHA256 val.jsonl:  ", sha256_file(val_path))
print(">>> CATAT kedua hash ini ke docs/finetune_recipe.md (harus sama dengan run gemma).")

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

train_data = load_jsonl(train_path)
val_data = load_jsonl(val_path)

print(f"\ntrain: {len(train_data)} contoh")
print(f"val:   {len(val_data)} contoh")

print("\n=== CONTOH ===")
print("INSTRUCTION:", train_data[0]["instruction"][:120], "...")
print("\nINPUT:\n", train_data[0]["input"][:200])
print("\nOUTPUT:\n", train_data[0]["output"][:300])

SHA256 train.jsonl: 65468a4c6ad296d17cdd0d8dd1b8b1f768fa0356cb48cbcee3f6541724511459
SHA256 val.jsonl:   fdb77aebe1238387a025185390649b6d2d5f8f8ce03df1f6fb1f1c9c051c398c
>>> CATAT kedua hash ini ke docs/finetune_recipe.md (harus sama dengan run gemma).

train: 194 contoh
val:   49 contoh

=== CONTOH ===
INSTRUCTION: Ubah statistik deterministik sebuah grafik konsumsi listrik rumah tangga menjadi narasi insight 3-5 kalimat dalam Bahasa ...

INPUT:
 chart_id: sub_metering_comparison
record_count: 2033491
avg_sub_metering_1_wh: 0,9635
avg_sub_metering_2_wh: 1,7508
avg_sub_metering_3_wh: 7,1857
total_sub_metering_1_wh: 1959268,6
total_sub_metering_

OUTPUT:
 Berdasarkan data konsumsi listrik rumah tangga, sub-metering ketiga mencatat rata-rata energi tertinggi sebesar 7,1857 Wh, jauh melampaui sub-metering pertama (0,9635 Wh) dan kedua (1,7508 Wh). Disparitas signifikan ini mengindikasikan bahwa beban dasar rumah tangga didominasi oleh peralatan yang te


## 3. Load Qwen2.5-1.5B-Instruct 4-bit (Dengan Penanganan Log & Peringatan)

Unsloth memuat base model terkuantisasi 4-bit — jauh lebih ringan dari gemma2-2b (1.5B vs 2.6B), muat sangat nyaman di T4, dan konsisten dengan model produksi (qwen2.5:1.5b Q4 di Ollama).

Versi library disamakan dengan run gemma (transformers 4.55.4 dst.) agar lingkungan training identik antar-kandidat. Peringatan-peringatan log diatasi secara terprogram tanpa melanggar konsistensi versi library.

In [ ]:
%%capture
# 1. Instalasi Unsloth jalur khusus Kaggle
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"

# 2. Pin versi utama (transformers & trl disamakan dengan gemma), peft di-upgrade untuk kompatibilitas Unsloth terbaru
!pip install --no-deps "transformers==4.55.4" "trl==0.21.0" "peft>=0.15.0" "tokenizers==0.21.2" "huggingface-hub<1.0"

In [2]:
import os
import warnings

# =========================================================================
# PENANGANAN PERINGATAN LOG (ANTI-STALL & CLEAN LOGS)
# =========================================================================
# 1. Mencegah error 403 / unduhan cepat macet (stall) akibat CDN Xet Hugging Face.
#    Memaksa Unsloth langsung memakai jalur normal Hugging Face Hub sejak awal.
os.environ["HF_HUB_DISABLE_XET"] = "1"

# 2. Menyembunyikan UserWarning terkait fitur fused-forward Unsloth yang meminta transformers >= 4.56.0.
#    Kita sengaja mengunci transformers di 4.55.4 demi validitas ilmiah kompetisi apple-to-apple dengan Gemma.
warnings.filterwarnings("ignore", message=".*fused-forward install skipped.*")
# =========================================================================

from unsloth import FastLanguageModel
import torch

max_seq_length = 1024   # sama dengan run gemma
dtype = None            # auto: float16 untuk T4
load_in_4bit = True     # QLoRA

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("Model & tokenizer dimuat dengan bersih tanpa stall CDN.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Model & tokenizer dimuat dengan bersih tanpa stall CDN.


### Sanity Check — Uji Base Model SEBELUM Training

In [3]:
# Verifikasi versi (harus transformers 4.55.x, sama dengan run gemma)
import transformers, trl
print("transformers:", transformers.__version__, "| trl:", trl.__version__)
assert transformers.__version__.startswith("4.55"), "STOP: versi transformers berbeda dari run gemma. Restart & cek instalasi."

# Sanity check: base model harus bisa bahasa waras sebelum di-train
FastLanguageModel.for_inference(model)
msgs = [{"role":"user","content":"Jelaskan dalam satu kalimat apa itu konsumsi listrik."}]
ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=ids, max_new_tokens=60, temperature=0.3, do_sample=True)
print("\n=== OUTPUT BASE MODEL ===")
print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))
print("\n>>> Jika output di atas WARAS (kalimat Indonesia normal), lanjut training.")
print(">>> Jika ACAK/gibberish, STOP — masalah versi, jangan lanjut.")
FastLanguageModel.for_training(model)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


transformers: 4.55.4 | trl: 0.21.0

=== OUTPUT BASE MODEL ===
Konsumsi listrik adalah jumlah energi yang dihasilkan oleh sumber daya listrik seperti tenaga alami atau fosil, dan kemudian digunakan untuk berbagai keperluan manusia, seperti memasak, menulis, dan bermain game.

>>> Jika output di atas WARAS (kalimat Indonesia normal), lanjut training.
>>> Jika ACAK/gibberish, STOP — masalah versi, jangan lanjut.


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536, padding_idx=151654)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((153

In [4]:
# Pasang LoRA adapter — konfigurasi IDENTIK dengan run gemma (apple-to-apple).
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                      # rank LoRA (sama dengan gemma)
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

# VERIFIKASI KRITIS: pastikan LoRA benar-benar menempel ke modul.
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} ({100*trainable/total:.2f}%)")
assert trainable > 1_000_000, (
    "STOP: LoRA TIDAK menempel (trainable params terlalu kecil). "
    "Cek log: jika 'patched ... 0 QKV layers', berarti versi library salah."
)
print("LoRA adapter terpasang dengan benar (>1M trainable params).")

Unsloth 2026.7.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Trainable params: 18,464,768 (2.04%)
LoRA adapter terpasang dengan benar (>1M trainable params).


## 4. Format Data dengan Chat Template (ChatML)

**Perbedaan utama dari gemma:** Qwen2.5 memakai format ChatML (`<|im_start|>user ... <|im_end|>`), bukan `<start_of_turn>`. Kode di bawah tetap memakai `tokenizer.apply_chat_template` sehingga format otomatis benar; yang perlu disesuaikan adalah penanda masking di Bagian 5.

In [7]:
from datasets import Dataset

def to_chat(example):
    user_msg = example["instruction"].strip() + "\n\n=== STATISTIK INPUT ===\n" + example["input"].strip()
    messages = [
        {"role":"user","content":user_msg},
        {"role":"assistant","content":example["output"].strip()},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

train_ds = Dataset.from_list(train_data).map(to_chat)
val_ds   = Dataset.from_list(val_data).map(to_chat)

print("=== CONTOH TEKS TERFORMAT (perhatikan penanda <|im_start|>) ===")
print(train_ds[0]["text"][:600])

Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Map:   0%|          | 0/49 [00:00<?, ? examples/s]

=== CONTOH TEKS TERFORMAT (perhatikan penanda <|im_start|>) ===
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Ubah statistik deterministik sebuah grafik konsumsi listrik rumah tangga menjadi narasi insight 3-5 kalimat dalam Bahasa Indonesia formal-teknis. Setiap angka harus berasal dari statistik input (atau turunannya seperti load factor dan koefisien variasi) dan disertai satuan baku; gunakan minimal dua istilah domain yang relevan tanpa mengarang angka.

=== STATISTIK INPUT ===
chart_id: sub_metering_comparison
record_count: 2033491
avg_sub_metering_1_wh: 0,9635
avg_sub_metering_2_wh: 


## 5. Training QLoRA

Konfigurasi IDENTIK dengan run gemma: 3 epoch, effective batch 8 (batch 2 × grad accum 4), LR 2e-4, seed 42. Untuk ~194 contoh perkiraan **beberapa menit** di T4 (1.5B lebih cepat dari 2.6B).

In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        save_strategy = "no",
        eval_strategy = "epoch",
        report_to = "none",
    ),
)

# === KRUSIAL: latih HANYA pada bagian jawaban (narasi), bukan bagian input ===
# PENANDA BERBEDA DARI GEMMA: Qwen2.5 memakai ChatML.
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)
print("train_on_responses_only aktif — loss hanya pada bagian narasi.")

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/194 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/49 [00:00<?, ? examples/s]

Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Map:   0%|          | 0/49 [00:00<?, ? examples/s]

train_on_responses_only aktif — loss hanya pada bagian narasi.


### Verifikasi Masking — pastikan hanya narasi yang dilatih

In [9]:
import numpy as np
ex = trainer.train_dataset[0]
labels = np.array(ex["labels"])
input_ids = np.array(ex["input_ids"])

trained_ids = input_ids[labels != -100]
masked_count = int((labels == -100).sum())
trained_count = int((labels != -100).sum())

print(f"Token di-mask (tidak dilatih / bagian input): {masked_count}")
print(f"Token dilatih (bagian narasi): {trained_count}")
assert trained_count > 0, "STOP: tidak ada token yang dilatih — penanda ChatML salah."
print("\n=== TEKS YANG DILATIH (harus HANYA narasi, bukan statistik input) ===")
print(tokenizer.decode(trained_ids, skip_special_tokens=False)[:600])
print("\n>>> Jika di atas hanya narasi Bahasa Indonesia, masking BENAR.")
print(">>> Jika muncul 'chart_id:' / 'STATISTIK INPUT' / angka statistik, masking SALAH.")

Token di-mask (tidak dilatih / bagian input): 279
Token dilatih (bagian narasi): 233

=== TEKS YANG DILATIH (harus HANYA narasi, bukan statistik input) ===
Berdasarkan data konsumsi listrik rumah tangga, sub-metering ketiga mencatat rata-rata energi tertinggi sebesar 7,1857 Wh, jauh melampaui sub-metering pertama (0,9635 Wh) dan kedua (1,7508 Wh). Disparitas signifikan ini mengindikasikan bahwa beban dasar rumah tangga didominasi oleh peralatan yang terhubung ke sub-metering ketiga, sementara sub-metering lainnya berkontribusi sebagai beban rendah yang lebih fluktuatif. Pola tersebut konsisten dengan profil beban residensial tipikal, di mana perangkat dengan daya tinggi seperti pemanas atau pompa menjadi kontributor utama energi total. Temuan ini

>>> Jika di atas hanya narasi Bahasa Indonesia, masking BENAR.
>>> Jika muncul 'chart_id:' / 'STATISTIK INPUT' / angka statistik, masking SALAH.


In [10]:
gpu_stats = torch.cuda.get_device_properties(0)
start_mem = round(torch.cuda.max_memory_reserved()/1024/1024/1024, 2)
max_mem = round(gpu_stats.total_memory/1024/1024/1024, 2)
print(f"GPU: {gpu_stats.name}, total {max_mem} GB")
print(f"Terpakai sebelum training: {start_mem} GB")

GPU: Tesla T4, total 14.56 GB
Terpakai sebelum training: 1.5 GB


In [11]:
trainer_stats = trainer.train()
print("\nTraining selesai. Loss akhir:", trainer_stats.metrics.get("train_loss"))

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 194 | Num Epochs = 3 | Total steps = 39
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,1.343100,1.020386
2,0.785500,0.763116
3,0.693700,0.698911


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient



Training selesai. Loss akhir: 1.0068460733462603


In [12]:
used_mem = round(torch.cuda.max_memory_reserved()/1024/1024/1024, 2)
print(f"Waktu training: {trainer_stats.metrics['train_runtime']:.0f} detik")
print(f"Puncak VRAM: {used_mem} GB / {max_mem} GB")

Waktu training: 278 detik
Puncak VRAM: 3.22 GB / 14.56 GB


## 6. Uji Inferensi — Sanity Check Anti-Halusinasi

Uji dengan statistik nyata YANG SAMA dengan sanity check gemma, agar output kedua narrator bisa dibandingkan langsung.

In [13]:
FastLanguageModel.for_inference(model)

# Input uji SAMA dengan notebook gemma (komparasi langsung).
test_input = '''chart_id: hourly_consumption_pattern
hours: 24
avg_global_active_power_kw: 1,091
min_hour: 04:00
min_value_kw: 0,444
max_hour: 20:00
max_value_kw: 1,899
unit: kW'''

instruction = train_data[0]["instruction"]
user_msg = instruction + "\n\n=== STATISTIK INPUT ===\n" + test_input

messages = [{"role":"user","content":user_msg}]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 256,
    temperature = 0.4,
    top_p = 0.9,
    do_sample = True,
)
result = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("=== NARASI HASIL FINE-TUNE (QWEN) ===\n")
print(result)

=== NARASI HASIL FINE-TUNE (QWEN) ===

Pemantauan beban aktif harian menunjukkan rata-rata daya aktif sebesar 1,091 kW dengan disparitas antara puncak dan lembah yang mencapai 1,899 kW pada jam pukul 20.00 dibandingkan 0,444 kW pada jam terendah 04.00. Load factor yang dihitung dari perbandingan rata-rata terhadap puncak adalah sekitar 0,47, mengindikasikan bahwa kurva beban residensial memiliki profil beban yang sangat fluktuatif. Pola ini konsisten dengan karakteristik aktivitas rumah tangga, sehingga berpotensi mendukung strategi demand response untuk meratakan lonjakan beban puncak melalui penggunaan energi saat low-tide.


**Checklist verifikasi manual (sama dengan gemma):**
- [ ] Semua angka (1,091 / 0,444 / 1,899 kW) muncul benar, tidak ada angka baru
- [ ] Satuan kW disebut
- [ ] Minimal 2 istilah domain (load factor, base load, kurva beban, dll)
- [ ] Tidak ada benchmark eksternal / p-value / prediksi kuantitatif karangan
- [ ] Bahasa Indonesia formal-teknis, mengalir

## 7. Simpan Adapter & Export GGUF Q4_K_M untuk Ollama

### 7a. Simpan adapter LoRA (cadangan ringan)

In [14]:
model.save_pretrained("qwen25-energy-insight-lora")
tokenizer.save_pretrained("qwen25-energy-insight-lora")
print("Adapter LoRA tersimpan di qwen25-energy-insight-lora/")

Adapter LoRA tersimpan di qwen25-energy-insight-lora/


### 7b. Export ke GGUF Q4_K_M (untuk Ollama lokal di GTX 1650)

Merge adapter ke base lalu kuantisasi Q4_K_M — kuantisasi yang SAMA dengan kandidat base di benchmark.

In [15]:
model.save_pretrained_gguf(
    "qwen25-energy-insight-gguf",
    tokenizer,
    quantization_method = "q4_k_m",
)
print("GGUF Q4_K_M tersimpan di qwen25-energy-insight-gguf/")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:09<00:00,  9.93s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:14<00:00, 14.84s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/qwen25-energy-insight-gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b9987-mix-53618c5 (app-b9987-mix-53618c5-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['qwen25-energy-insight-gguf_gguf/Qwen2.5-1.5B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: Al

In [16]:
# Lihat file GGUF + catat SHA256-nya (untuk configs/models/*.yaml & finetune_recipe.md)
import glob
found = glob.glob("/kaggle/working/**/*.gguf", recursive=True)
for p in found:
    sz = os.path.getsize(p)/1024/1024
    print(f"{p}  ({sz:.0f} MB)")
    print("  SHA256:", sha256_file(p))
print(">>> CATAT hash GGUF ini ke docs/finetune_recipe.md.")

/kaggle/working/qwen25-energy-insight-gguf_gguf/Qwen2.5-1.5B-Instruct.Q4_K_M.gguf  (940 MB)
  SHA256: a8bfe84a09960c94b6cda4fbf8399cec8be25231f761005a833911af14e7be50
>>> CATAT hash GGUF ini ke docs/finetune_recipe.md.


### 7c. Download GGUF ke komputer

In [17]:
from IPython.display import FileLink
for f in glob.glob("/kaggle/working/**/*.gguf", recursive=True):
    display(FileLink(f))

/kaggle/working/qwen25-energy-insight-gguf_gguf/Qwen2.5-1.5B-Instruct.Q4_K_M.gguf

## 8. Deploy di Ollama Lokal (jalankan di GTX 1650, bukan di Kaggle)

Buat `Modelfile` dengan **system prompt & parameter IDENTIK dengan deploy gemma**:

```
FROM ./qwen25-energy-insight-q4_k_m.gguf
PARAMETER temperature 0.4
PARAMETER top_p 0.9
PARAMETER num_gpu 99
SYSTEM """Anda adalah analis data energi yang menyusun narasi insight dari statistik konsumsi listrik rumah tangga. Setiap angka harus berasal dari statistik input dan disertai satuan baku. Gunakan istilah domain yang relevan tanpa mengarang angka, benchmark, atau prediksi."""
```

Lalu:
```bash
ollama create qwen25-energy-insight:v1 -f Modelfile
ollama run qwen25-energy-insight:v1
```

**Integrasi ke proyek:** ganti narrator via env — `OLLAMA_INSIGHT_MODEL=qwen25-energy-insight:v1` di `.env` (mekanisme `insight_model` di `configs/model.yaml`; SQL/planner/reporter tetap di model utama).

## 9. Evaluasi Narasi Report (Bab 4)

Tiga lapis, apple-to-apple antara `gemma2-energy-insight:v3` vs `qwen25-energy-insight:v1` pada stat block yang sama:
1. **Checker deterministik existing** — `unit_rule_compliance`, `numeric_fact_coverage` (skrip evaluasi report proyek).
2. **Rubrik manual dua penilai** (subset 20–30 narasi) — untuk memvalidasi judge.
3. **LLM-as-a-judge** — protokol lengkap di `docs/llm_judge_report_eval_plan.md`.